In [1]:
# ============================================================
# 🔥 Netflix Churn Prediction — PyTorch (PIPELINE-COMPATIBLE)
# Production-grade • Fairness-ready • InferStream-compatible
# ============================================================

import pandas as pd
import numpy as np
import os
import json
import joblib
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime

# ================================
# Config
# ================================
DATA_PATH = "../data/netflix_customer_churn.csv"
MODEL_DIR = "../backend/models/netflix"

MODEL_NAME = "pytorch_model.pt"
SCALER_NAME = "scaler_pytorch.pkl"
ENCODER_NAME = "encoder_pytorch.pkl"
FEATURES_NAME = "feature_names_pytorch.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# ================================
# Load Data
# ================================
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded dataset: {df.shape}")

TARGET_COL = "churned"
df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(int)

# Drop identifiers
for col in ["customer_id", "customerid", "user_id"]:
    if col in df.columns:
        df = df.drop(columns=[col])

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# ================================
# Feature Groups
# ================================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("🔢 Numeric features:", num_cols)
print("🏷️ Categorical features:", cat_cols)

# ================================
# Preprocessing
# ================================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out().tolist()

# Save preprocessing artifacts
joblib.dump(
    preprocessor.named_transformers_["num"],
    os.path.join(MODEL_DIR, SCALER_NAME)
)
joblib.dump(
    preprocessor.named_transformers_["cat"],
    os.path.join(MODEL_DIR, ENCODER_NAME)
)
with open(os.path.join(MODEL_DIR, FEATURES_NAME), "w") as f:
    json.dump(feature_names, f, indent=2)

# ================================
# Train/Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

# ================================
# Define Model
# ================================
class ChurnNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = ChurnNet(X_train.shape[1])
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ================================
# Train
# ================================
EPOCHS = 15
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb).squeeze()
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

# ================================
# Evaluate
# ================================
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb).squeeze()
        preds = (preds > 0.5).int()
        all_preds.extend(preds.tolist())
        all_labels.extend(yb.tolist())

print("📊 Classification Report:")
print(classification_report(all_labels, all_preds, digits=4))
print("🧮 Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

# ================================
# Save Model
# ================================
torch.save(model.state_dict(), os.path.join(MODEL_DIR, MODEL_NAME))

print("✅ Saved artifacts:")
print(f"   • {MODEL_NAME}")
print(f"   • {SCALER_NAME}")
print(f"   • {ENCODER_NAME}")
print(f"   • {FEATURES_NAME}")

print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

✅ Loaded dataset: (5000, 14)
🔢 Numeric features: ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']
🏷️ Categorical features: ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']


/var/folders/91/syqbgxdn69n01td9pvbx3dz80000gn/T/ipykernel_27054/3057883809.py:56: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()


Epoch 1/15, Loss: 40.1811
Epoch 2/15, Loss: 23.7237
Epoch 3/15, Loss: 15.8084
Epoch 4/15, Loss: 14.7186
Epoch 5/15, Loss: 14.7308
Epoch 6/15, Loss: 14.1634
Epoch 7/15, Loss: 13.7063
Epoch 8/15, Loss: 13.3950
Epoch 9/15, Loss: 13.1790
Epoch 10/15, Loss: 12.8124
Epoch 11/15, Loss: 12.4837
Epoch 12/15, Loss: 12.3843
Epoch 13/15, Loss: 12.2946
Epoch 14/15, Loss: 12.0823
Epoch 15/15, Loss: 11.9690
📊 Classification Report:
              precision    recall  f1-score   support

         0.0     0.9258    0.9034    0.9145       497
         1.0     0.9068    0.9284    0.9175       503

    accuracy                         0.9160      1000
   macro avg     0.9163    0.9159    0.9160      1000
weighted avg     0.9162    0.9160    0.9160      1000

🧮 Confusion Matrix:
[[449  48]
 [ 36 467]]
✅ Saved artifacts:
   • pytorch_model.pt
   • scaler_pytorch.pkl
   • encoder_pytorch.pkl
   • feature_names_pytorch.json
🏁 Done at 2026-01-24 18:48:17
